In [1]:
# Assignment 3 - K-Means Clustering

import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from scipy.stats import chi2_contingency
from itertools import combinations

In [2]:
## 1. Load Expression Data and Metadata

df = pd.read_csv("../data/SRP100712.tsv", sep="\t")

meta = pd.read_csv("../data/metadata_SRP100712.tsv", sep="\t")
meta = meta.set_index("refinebio_accession_code")

print("Raw expression shape:", df.shape)
print("Metadata shape:", meta.shape)

Raw expression shape: (32309, 595)
Metadata shape: (594, 23)


In [3]:
df.head()

,Gene,SRR5285467,SRR5285468,SRR5285469,SRR5285470,SRR5285471,SRR5285472,SRR5285473,SRR5285475,SRR5285476,...,SRR5286096,SRR5286097,SRR5286098,SRR5286100,SRR5286101,SRR5286102,SRR5286103,SRR5286104,SRR5286105,SRR5286106
0,AT1G01010,12.807026,10.719908,6.809609,6.802937,9.705004,5.903112,10.698493,5.896903,6.883874,...,7.872030,5.904026,12.834161,5.911054,13.853484,8.912395,11.887026,3.961550,3.971817,6.888621
1,AT1G01020,31.309752,23.725655,22.808340,24.743242,14.957870,22.627356,32.477890,21.267132,19.228527,...,29.608995,35.108727,36.803383,47.853720,38.696342,45.518707,40.453710,46.531265,45.546875,29.633783
2,AT1G01030,1.997912,0.000000,0.989656,0.000000,0.961891,0.996533,0.986524,1.026043,1.000709,...,1.973204,3.949674,2.969075,2.969002,4.954153,2.980675,3.847060,4.803059,3.980706,2.975284
3,AT1G01040,9.554996,5.177971,4.798006,1.035365,1.914409,3.836754,2.875084,1.219955,2.066959,...,4.894994,3.173866,11.648278,11.668991,8.775879,7.831170,10.763938,9.780508,4.897013,8.814977
4,AT1G01046,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.867567,0.000000,0.000000,0.000000,0.000000


In [4]:
meta = meta.loc[df.drop(columns=["Gene"]).columns]

meta["condition"] = np.where(
    meta["refinebio_subject"].str.contains("spaceflight", case=False, na=False),
    "flight",
    "ground"
)

meta["age"] = meta["refinebio_age"].astype(int).astype(str) + "d"

print("Unique biological sample titles:", meta["refinebio_title"].nunique())
print("\nCondition counts:")
print(meta["condition"].value_counts())

Unique biological sample titles: 32

Condition counts:
condition
ground    309
flight    285
Name: count, dtype: int64


In [5]:
## Process Expression Data

drop_gene = df.drop(columns=["Gene"])
drop_gene.index = df["Gene"]

drop_gene = drop_gene.T.groupby(meta["refinebio_title"]).sum().T

print("Aggregated expression shape:", drop_gene.shape)

Aggregated expression shape: (32309, 32)


In [6]:
log_expr = np.log2(drop_gene + 1)

print("Log-transformed expression shape:", log_expr.shape)

Log-transformed expression shape: (32309, 32)


In [7]:
expression = log_expr.T.copy()

print("Clustering matrix shape:", expression.shape)

Clustering matrix shape: (32, 32309)


In [8]:
sample_meta = meta.groupby("refinebio_title")[["condition", "age"]].first()

# Keep exactly the same sample order as the expression matrix
sample_meta = sample_meta.loc[expression.index]

# Add genotype from the sample title
sample_meta["genotype"] = np.where(
    sample_meta.index.str.contains("Col-0", case=False),
    "Col-0",
    "WS"
)

sample_meta.head()

,condition,age,genotype
refinebio_title,,,
"Ground Control, 4 days old, Col-0 Rep1",ground,4d,Col-0
"Ground Control, 4 days old, Col-0 Rep2",ground,4d,Col-0
"Ground Control, 4 days old, Col-0 Rep3",ground,4d,Col-0
"Ground Control, 4 days old, Col-0 Rep4",ground,4d,Col-0
"Ground Control, 4 days old, WS Rep1",ground,4d,WS


In [9]:
print("Expression samples:", expression.shape[0])
print("Metadata samples:", sample_meta.shape[0])

print("\nCondition:")
print(sample_meta["condition"].value_counts())

print("\nGenotype:")
print(sample_meta["genotype"].value_counts())

print("\nAge:")
print(sample_meta["age"].value_counts())

Expression samples: 32
Metadata samples: 32

Condition:
condition
ground    16
flight    16
Name: count, dtype: int64

Genotype:
genotype
Col-0    16
WS       16
Name: count, dtype: int64

Age:
age
4d    16
8d    16
Name: count, dtype: int64


In [10]:
## Save Shared Processed Data

expression.to_csv("shared_expression_log2.csv")
sample_meta.to_csv("shared_sample_metadata.csv")

print("Shared files saved.")

Shared files saved.


In [11]:
## 2. Unsupervised Analysis

gene_variance = expression.var(axis=0)

print("Number of genes ranked:", len(gene_variance))

gene_variance.head()

Number of genes ranked: 32309


Gene
AT1G01010    0.738825
AT1G01020    0.486446
AT1G01030    0.699102
AT1G01040    0.469311
AT1G01046    0.374836
dtype: float64

In [12]:
top_5000_genes = gene_variance.nlargest(5000).index

expression_5000 = expression.loc[:, top_5000_genes]

print("Full expression matrix:", expression.shape)
print("Top 5,000 matrix:", expression_5000.shape)

Full expression matrix: (32, 32309)
Top 5,000 matrix: (32, 5000)


In [13]:
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

sample_meta["kmeans_k2"] = kmeans.fit_predict(expression_5000)

sample_meta[["condition", "genotype", "age", "kmeans_k2"]].head(10)

c:\Users\marco\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\marco\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\marco\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\marco\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 1036

,condition,genotype,age,kmeans_k2
refinebio_title,,,,
"Ground Control, 4 days old, Col-0 Rep1",ground,Col-0,4d,0
"Ground Control, 4 days old, Col-0 Rep2",ground,Col-0,4d,0
"Ground Control, 4 days old, Col-0 Rep3",ground,Col-0,4d,0
"Ground Control, 4 days old, Col-0 Rep4",ground,Col-0,4d,0
"Ground Control, 4 days old, WS Rep1",ground,WS,4d,1
"Ground Control, 4 days old, WS Rep2",ground,WS,4d,1
"Ground Control, 4 days old, WS Rep3",ground,WS,4d,1
"Ground Control, 4 days old, WS Rep4",ground,WS,4d,1
"Ground Control, 8 days old, Col-0 Rep1",ground,Col-0,8d,0


In [14]:
sample_meta["kmeans_k2"].value_counts().sort_index()

kmeans_k2
0    16
1    16
Name: count, dtype: int64

In [15]:
pd.crosstab(
    sample_meta["condition"],
    sample_meta["kmeans_k2"]
)

kmeans_k2,0,1
condition,,
flight,8,8
ground,8,8


In [16]:
pd.crosstab(
    sample_meta["genotype"],
    sample_meta["kmeans_k2"]
)

kmeans_k2,0,1
genotype,,
Col-0,16,0
WS,0,16


In [17]:
### Effect of Changing k

for k in [2, 3, 4, 5]:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    sample_meta[f"kmeans_k{k}"] = model.fit_predict(expression_5000)

In [18]:
k_columns = [
    "kmeans_k2",
    "kmeans_k3",
    "kmeans_k4",
    "kmeans_k5"
]

sample_meta[
    ["condition", "genotype", "age"] + k_columns
].head(15)

,condition,genotype,age,kmeans_k2,kmeans_k3,kmeans_k4,kmeans_k5
refinebio_title,,,,,,,
"Ground Control, 4 days old, Col-0 Rep1",ground,Col-0,4d,0,1,1,0
"Ground Control, 4 days old, Col-0 Rep2",ground,Col-0,4d,0,1,1,0
"Ground Control, 4 days old, Col-0 Rep3",ground,Col-0,4d,0,1,1,0
"Ground Control, 4 days old, Col-0 Rep4",ground,Col-0,4d,0,1,1,0
"Ground Control, 4 days old, WS Rep1",ground,WS,4d,1,2,3,1
"Ground Control, 4 days old, WS Rep2",ground,WS,4d,1,0,3,1
"Ground Control, 4 days old, WS Rep3",ground,WS,4d,1,0,3,1
"Ground Control, 4 days old, WS Rep4",ground,WS,4d,1,2,3,1
"Ground Control, 8 days old, Col-0 Rep1",ground,Col-0,8d,0,1,1,0


In [19]:
for k in [2, 3, 4, 5]:
    print(f"\nk = {k}")
    print(sample_meta[f"kmeans_k{k}"].value_counts().sort_index())


k = 2
kmeans_k2
0    16
1    16
Name: count, dtype: int64

k = 3
kmeans_k3
0     7
1    15
2    10
Name: count, dtype: int64

k = 4
kmeans_k4
0     8
1    15
2     1
3     8
Name: count, dtype: int64

k = 5
kmeans_k5
0    15
1     6
2     8
3     1
4     2
Name: count, dtype: int64


In [20]:
### Effect of the Number of Genes

gene_counts = [10, 100, 1000, 5000, 10000]

for n_genes in gene_counts:

    top_genes = gene_variance.nlargest(n_genes).index

    subset = expression.loc[:, top_genes]

    model = KMeans(
        n_clusters=2,
        random_state=42,
        n_init=10
    )

    sample_meta[f"kmeans_{n_genes}_genes"] = model.fit_predict(subset)

In [21]:
gene_cluster_columns = [
    "kmeans_10_genes",
    "kmeans_100_genes",
    "kmeans_1000_genes",
    "kmeans_5000_genes",
    "kmeans_10000_genes"
]

sample_meta[
    ["condition", "genotype", "age"] + gene_cluster_columns
].head(15)

,condition,genotype,age,kmeans_10_genes,kmeans_100_genes,kmeans_1000_genes,kmeans_5000_genes,kmeans_10000_genes
refinebio_title,,,,,,,,
"Ground Control, 4 days old, Col-0 Rep1",ground,Col-0,4d,0,0,0,0,0
"Ground Control, 4 days old, Col-0 Rep2",ground,Col-0,4d,0,0,0,0,0
"Ground Control, 4 days old, Col-0 Rep3",ground,Col-0,4d,0,0,0,0,0
"Ground Control, 4 days old, Col-0 Rep4",ground,Col-0,4d,0,0,0,0,0
"Ground Control, 4 days old, WS Rep1",ground,WS,4d,1,1,1,1,1
"Ground Control, 4 days old, WS Rep2",ground,WS,4d,1,1,1,1,1
"Ground Control, 4 days old, WS Rep3",ground,WS,4d,1,1,1,1,1
"Ground Control, 4 days old, WS Rep4",ground,WS,4d,1,1,1,1,1
"Ground Control, 8 days old, Col-0 Rep1",ground,Col-0,8d,0,0,0,0,0


In [22]:
for column in gene_cluster_columns:
    print(f"\n{column}")
    print(sample_meta[column].value_counts().sort_index())


kmeans_10_genes
kmeans_10_genes
0    16
1    16
Name: count, dtype: int64

kmeans_100_genes
kmeans_100_genes
0    16
1    16
Name: count, dtype: int64

kmeans_1000_genes
kmeans_1000_genes
0    16
1    16
Name: count, dtype: int64

kmeans_5000_genes
kmeans_5000_genes
0    16
1    16
Name: count, dtype: int64

kmeans_10000_genes
kmeans_10000_genes
0    15
1    17
Name: count, dtype: int64


In [23]:
### Chi-Squared Comparison of Gene-Count Results

chi_square_results = []

for col1, col2 in combinations(gene_cluster_columns, 2):

    contingency = pd.crosstab(
        sample_meta[col1],
        sample_meta[col2]
    )

    chi2, p_value, dof, expected = chi2_contingency(contingency)

    chi_square_results.append({
        "Clustering_1": col1,
        "Clustering_2": col2,
        "Chi_Square": chi2,
        "Degrees_of_Freedom": dof,
        "P_Value": p_value
    })

chi_square_df = pd.DataFrame(chi_square_results)

chi_square_df

,Clustering_1,Clustering_2,Chi_Square,Degrees_of_Freedom,P_Value
0,kmeans_10_genes,kmeans_100_genes,28.125000,1,1.137273e-07
1,kmeans_10_genes,kmeans_1000_genes,28.125000,1,1.137273e-07
2,kmeans_10_genes,kmeans_5000_genes,28.125000,1,1.137273e-07
3,kmeans_10_genes,kmeans_10000_genes,24.596078,1,7.069546e-07
4,kmeans_100_genes,kmeans_1000_genes,28.125000,1,1.137273e-07
5,kmeans_100_genes,kmeans_5000_genes,28.125000,1,1.137273e-07
6,kmeans_100_genes,kmeans_10000_genes,24.596078,1,7.069546e-07
7,kmeans_1000_genes,kmeans_5000_genes,28.125000,1,1.137273e-07
8,kmeans_1000_genes,kmeans_10000_genes,24.596078,1,7.069546e-07
9,kmeans_5000_genes,kmeans_10000_genes,24.596078,1,7.069546e-07


In [24]:
from statsmodels.stats.multitest import multipletests

chi_square_df["Adjusted_P_Value"] = multipletests(
    chi_square_df["P_Value"],
    method="fdr_bh"
)[1]

chi_square_df

,Clustering_1,Clustering_2,Chi_Square,Degrees_of_Freedom,P_Value,Adjusted_P_Value
0,kmeans_10_genes,kmeans_100_genes,28.125000,1,1.137273e-07,1.895454e-07
1,kmeans_10_genes,kmeans_1000_genes,28.125000,1,1.137273e-07,1.895454e-07
2,kmeans_10_genes,kmeans_5000_genes,28.125000,1,1.137273e-07,1.895454e-07
3,kmeans_10_genes,kmeans_10000_genes,24.596078,1,7.069546e-07,7.069546e-07
4,kmeans_100_genes,kmeans_1000_genes,28.125000,1,1.137273e-07,1.895454e-07
5,kmeans_100_genes,kmeans_5000_genes,28.125000,1,1.137273e-07,1.895454e-07
6,kmeans_100_genes,kmeans_10000_genes,24.596078,1,7.069546e-07,7.069546e-07
7,kmeans_1000_genes,kmeans_5000_genes,28.125000,1,1.137273e-07,1.895454e-07
8,kmeans_1000_genes,kmeans_10000_genes,24.596078,1,7.069546e-07,7.069546e-07
9,kmeans_5000_genes,kmeans_10000_genes,24.596078,1,7.069546e-07,7.069546e-07


In [25]:
## Save K-Means Results

sample_meta.to_csv(
    "results/kmeans_cluster_assignments.csv"
)

chi_square_df.to_csv(
    "results/kmeans_gene_count_chi_square.csv",
    index=False
)

print("K-means results saved.")

K-means results saved.


In [26]:
summary_rows = []

for n_genes in gene_counts:

    column = f"kmeans_{n_genes}_genes"

    counts = sample_meta[column].value_counts().sort_index()

    summary_rows.append({
        "Gene_Count": n_genes,
        "Number_of_Clusters": sample_meta[column].nunique(),
        "Cluster_0_Size": counts.get(0, 0),
        "Cluster_1_Size": counts.get(1, 0)
    })

summary_table = pd.DataFrame(summary_rows)

summary_table

,Gene_Count,Number_of_Clusters,Cluster_0_Size,Cluster_1_Size
0,10,2,16,16
1,100,2,16,16
2,1000,2,16,16
3,5000,2,16,16
4,10000,2,15,17


In [27]:
summary_table.to_csv(
    "results/kmeans_gene_count_summary.csv",
    index=False
)